In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

## Employees


In [0]:
df_emp = spark.read.table('project_cata.bronze.employees')
# df_emp.printSchema()
# df_nulls = df_emp.select([sum(when(col(i).isNull() | (trim(col(i)) == ""), 1).otherwise(0)).alias(i) for i in df_emp.columns])
# display(df_nulls)

# converting date to valid date formats
formats = ["yyyy-MM-dd", "dd/MM/yyyy", "MM/dd/yyyy", "dd-MM-yyyy"]
df_emp = df_emp.withColumn('date_of_birth', coalesce(*(try_to_date(col('date_of_birth'), f) for f in formats))).withColumn('hire_date', coalesce(*(try_to_date(col('hire_date'), f) for f in formats))).withColumn('last_updated', to_timestamp(col("last_updated"), "yyyy-MM-dd HH:mm:ss"))

#removing duplicates
df_emp = df_emp.withColumn('rono', row_number().over(Window.partitionBy('employee_id').orderBy(col('last_updated').desc())))
df_emp = df_emp.filter(col('rono') == 1).drop('rono')
# df_dups = df_emp.groupBy('employee_id').agg(count('employee_id').alias('ecount')).filter(col('ecount')>1)
# display(df_dups)

#Replacing invalid emails with Nulls
eregex = r'^[A-Za-z0-9_.-]+@[A-Za-z0-9_.-]+\.[A-Za-z0-9_]{2,}$'
df_emp = df_emp.withColumn('email', when(col('email').rlike(eregex), col('email')).otherwise(None))
# display(df_emp.filter(col('email').isNull()))

#checking any employees without department
df_dep = spark.read.table('project_cata.bronze.departments')
df_dep = df_dep.withColumn('sub_departments', explode(col('sub_departments.department'))).withColumn('sub_dep_id', col('sub_departments._id')).withColumn('sub_dep_name', col('sub_departments.name'))

df_emp = df_emp.join(df_dep.select('sub_dep_id'), df_emp.department_id == df_dep.sub_dep_id, 'left').withColumn('reject_reason', when(col('sub_dep_id').isNull(), 'No Dep Id').when(col('hire_date').isNull(), 'No hire date').otherwise(None)).drop('sub_dep_id')

#Converting salry to float
df_emp = df_emp.withColumn('base_salary', round( regexp_replace(col('base_salary'), ',' , '').try_cast('float'), 2))

quatantine = df_emp.filter(col('reject_reason').isNotNull())
df_emp = df_emp.filter(col('reject_reason').isNull())

# Insert data to silver Employees
from delta.tables import DeltaTable

if not spark.catalog.tableExists('project_cata.silver.employees'):
    df_emp.write.format('delta').saveAsTable('project_cata.silver.employees')
else:
    target = DeltaTable.forName(spark , 'project_cata.silver.employees')
    target.alias("t").merge(df_emp.alias("s"), 't.employee_id=s.employee_id')\
        .whenMatchedUpdateAll(condition='s.last_updated >= t.last_updated ')\
            .whenNotMatchedInsertAll()\
                .execute()

In [0]:
%sql
-- drop table  project_cata.silver.employees
select * from  project_cata.silver.employees